# 02 — eModel Optimisation

Build an `EModelOptimizationScanConfig`, expand it via `GridScanGenerationTask`, and
run the `EModelOptimizationTask` for each coordinate. The task seeds its working
directory from the previous stage's output, merges the optimisation-related
`pipeline_settings` from the blocks into `recipes.json`, then runs `pipeline.optimise(seed=...)`.

**Reads from:** `obi-output/01_efeature_extraction/grid_scan/0/`.

**Writes to:** `obi-output/02_emodel_optimization/grid_scan/0/` — the working directory
plus the new `checkpoints/` directory and the `run/<iteration_tag>/` archive.

The example uses **`optimiser='CMA_ES'`** with very small `max_ngen=2` and
`offspring_size=4`. Increase those for production runs.


## Imports

In [ ]:
from pathlib import Path

import obi_one as obi
from obi_one.scientific.tasks.emodel_optimization._02_emodel_optimization.blocks import (
    OptimizationInitialize,
    OptimizationParams,
    OptimizationSettings,
)


## Build the scan config

In [ ]:
previous_stage = (
    Path.cwd() / "../../../obi-output/01_efeature_extraction/grid_scan/0"
).resolve()
assert previous_stage.exists(), (
    f"{previous_stage} not found — run 01_efeature_extraction.ipynb first."
)

scan_config = obi.EModelOptimizationScanConfig(
    initialize=OptimizationInitialize(
        previous_stage_output_path=previous_stage,
        emodel="L5PC",
        species="rat",
        brain_region="SSCX",
    ),
    optimization_settings=OptimizationSettings(
        optimiser="CMA_ES",
        max_ngen=2,            # very small for demo runs
        optimisation_timeout=300.0,
        validation_threshold=5.0,
        seed=1,
    ),
    optimization_params=OptimizationParams(
        offspring_size=4,      # very small for demo runs
    ),
)
print(scan_config.optimization_settings.to_dict(scan_config.optimization_params))


## Run the grid scan

In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root="../../../obi-output/02_emodel_optimization/grid_scan",
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

obi.run_tasks_for_generated_scan(grid_scan)


## Inspect the checkpoints

In [ ]:
coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print("Coordinate output:", coord_root)

checkpoints = sorted((coord_root / "checkpoints").rglob("*.pkl"))
print(f"Checkpoint files ({len(checkpoints)}):")
for p in checkpoints:
    print(" ", p.relative_to(coord_root))
